# Reproducing the fairness figure and policy table from published aggregates

This notebook redraws the four-panel price-of-fairness figure and rebuilds the policy scenario table using only the aggregate grid that the public site is built from. It does not re-solve the optimization and it does not touch child-level microdata, which are intentionally not published.

Scope and honest limits:
- The numbers come from results/phase3_fairness_grid.csv, the same aggregate file behind the live figure.
- The reproduction matches the published data and panel structure, not pixel-level styling of the original PNG.
- MICS 2019 microdata cannot be redistributed here, so this demo verifies the published aggregates rather than recomputing them from raw data.


In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/rafi-math/edu-support-optimization-bd.git',REPO], check=True)

RES = os.path.join(REPO,'results')
print('results/ contents:', sorted(os.listdir(RES)))

GRID = os.path.join(RES,'phase3_fairness_grid.csv')
if not os.path.isfile(GRID):
    raise RuntimeError('Missing '+GRID+' . Run: git ls-files results/  and send me the output.')


## Anchors

Before plotting, the notebook checks the published anchor values so a reader can see the figure is tied to verified numbers: raw objective 325.663745, raw observed out-of-school captured 332 of 641, proportional-young captured 219, and the age-gender interaction -5.740452 at the proportional corner.


In [ ]:
K = 730
TOTAL_OOS = 641
OBJ_RAW = 325.663745
CAP_RAW = 332
OOS_POOL_PARITY = 0.374415
POP_PARITY = 0.502740

g = pd.read_csv(GRID)
for c in ['young_floor','girls_floor','obj_unweighted','obj_weighted','observed_oos_captured','young_selected','girls_selected']:
    g[c] = pd.to_numeric(g[c], errors='coerce')
g['objective'] = g['objective'].astype(str).str.strip()

def cell(obj,y,gr):
    r = g[(g.objective==obj)&(g.young_floor==y)&(g.girls_floor==gr)]
    if len(r)!=1:
        raise RuntimeError('expected 1 cell for '+obj+' y='+str(y)+' g='+str(gr)+', got '+str(len(r)))
    return r.iloc[0]

raw = cell('unweighted',0,0)
prop = cell('unweighted',423,0)
gp = cell('unweighted',0,367)
corner = cell('unweighted',423,367)

assert abs(raw.obj_unweighted-OBJ_RAW)<1e-3
assert int(raw.observed_oos_captured)==CAP_RAW
assert int(prop.observed_oos_captured)==219

c_young = OBJ_RAW-prop.obj_unweighted
c_girls = OBJ_RAW-gp.obj_unweighted
c_both = OBJ_RAW-corner.obj_unweighted
interaction = c_both-c_young-c_girls
assert abs(interaction+5.740452)<1e-3

print('raw obj', round(raw.obj_unweighted,6), 'capture', int(raw.observed_oos_captured))
print('prop-young capture', int(prop.observed_oos_captured), 'cost', round(c_young,3))
print('interaction', round(interaction,6))


## Figure

Panels A and B price the under-13 floor for the unweighted objective with no girls floor. Panels C and D price the girls floor, shown with and without the proportional under-13 floor, against the same two grey parity references used on the live site (37.4% out-of-school-pool parity, 50.3% population parity).


In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(2,2, figsize=(11,8))
uw0 = g[(g.objective=='unweighted')&(g.girls_floor==0)].sort_values('young_floor')

ax[0,0].plot(uw0.young_floor/K, OBJ_RAW-uw0.obj_unweighted, marker='o', color='#0072B2')
ax[0,0].set_title('A. Cost of age coverage')
ax[0,0].set_xlabel('Selected share aged < 13')
ax[0,0].set_ylabel('Objective drop vs raw')
ax[0,0].annotate('proportional young floor 57.9% cost '+str(round(c_young,1)), xy=(423/K,c_young), xytext=(0.10,c_young*0.80), arrowprops=dict(arrowstyle='-',color='gray'))

ax[0,1].plot(uw0.young_floor/K, CAP_RAW-uw0.observed_oos_captured, marker='s', color='#D55E00')
ax[0,1].set_title('B. Reach cost of age coverage')
ax[0,1].set_xlabel('Selected share aged < 13')
ax[0,1].set_ylabel('Observed OOS not captured')
ax[0,1].annotate('proportional young floor loss '+str(CAP_RAW-int(prop.observed_oos_captured))+' observed OOS', xy=(423/K,CAP_RAW-prop.observed_oos_captured), xytext=(0.10,(CAP_RAW-prop.observed_oos_captured)*0.75), arrowprops=dict(arrowstyle='-',color='gray'))

for yf,col,mk,ls,lab in [(0,'#0072B2','o','-','No young floor'),(423,'#009E73','^','--','Proportional young floor')]:
    sub = g[(g.objective=='unweighted')&(g.young_floor==yf)].sort_values('girls_floor')
    base_o = float(sub[sub.girls_floor==0].obj_unweighted.iloc[0])
    base_c = float(sub[sub.girls_floor==0].observed_oos_captured.iloc[0])
    ax[1,0].plot(sub.girls_floor/K, base_o-sub.obj_unweighted, marker=mk, linestyle=ls, color=col, label=lab)
    ax[1,1].plot(sub.girls_floor/K, base_c-sub.observed_oos_captured, marker=mk, linestyle=ls, color=col, label=lab)

handles, labels = ax[1,0].get_legend_handles_labels()
handles = handles+[Line2D([0],[0],ls=':',color='gray'), Line2D([0],[0],ls='--',color='gray')]
labels = labels+['OOS-pool parity (37.4%)','Population parity (50.3%)']

for a in [ax[1,0],ax[1,1]]:
    a.axvline(OOS_POOL_PARITY, ls=':', color='gray', lw=0.8)
    a.axvline(POP_PARITY, ls='--', color='gray', lw=0.8)
    a.set_xlabel('Selected share girls')
    a.set_xlim(0.18,0.52)
    a.legend(handles=handles, labels=labels, fontsize=8)

ax[1,0].set_title('C. Cost of gender coverage')
ax[1,0].set_ylabel('Incremental objective drop')
ax[1,1].set_title('D. Reach cost of gender coverage')
ax[1,1].set_ylabel('Incremental observed OOS not captured')

fig.suptitle('Price of fairness constraints, rural Chattogram ages 6-17 (reproduced from published grid)')
fig.tight_layout()
plt.savefig('/content/phase3_fairness_reproduced.png', dpi=200)
plt.show()
print('Saved /content/phase3_fairness_reproduced.png')


## Policy table

The same grid yields the policy scenario table shown on the live site. The objective is the sum of out-of-fold predicted risk; observed out-of-school capture is an evaluation metric, not the optimization target. The interaction column is reported only where both floors are present.


In [ ]:
scen = [('raw risk no floors',0,0),('young 20pct only',146,0),('young proportional only',423,0),('girls 40pct only',0,292),('girls parity only',0,367),('young proportional + girls parity',423,367)]
rows = []
for name,y,gr in scen:
    r = cell('unweighted',y,gr)
    rw = cell('weighted',y,gr)
    rows.append({'policy':name,'young_floor':y,'girls_floor':gr,'obj_unweighted':round(float(r.obj_unweighted),2),'obj_drop_vs_raw':round(OBJ_RAW-float(r.obj_unweighted),2),'observed_oos_captured':int(r.observed_oos_captured),'capture_rate':round(int(r.observed_oos_captured)/TOTAL_OOS,4),'capture_drop_vs_raw':CAP_RAW-int(r.observed_oos_captured),'young_share':round(int(r.young_selected)/K,4),'girls_share':round(int(r.girls_selected)/K,4),'obj_weighted_sensitivity':round(float(rw.obj_weighted),2)})
pol = pd.DataFrame(rows)
print(pol.to_string(index=False))


## What this demonstrates

A reader can run this notebook end to end on free Colab with no data upload and no microdata, and recover the same anchors and the same four-panel structure that the public site publishes. That is the reproducibility claim: the published figure and table are derivable from a single committed aggregate file. The optimization itself, and the out-of-fold scores behind it, are documented in the repository scripts and are not re-run here because the underlying MICS microdata are not redistributable.

Live site: https://rafi-math.github.io/edu-support-optimization-bd/
Repository: https://github.com/rafi-math/edu-support-optimization-bd
